# Model 3 Training: Transformer (Restormer) for Image Restoration

This notebook trains Model 3: Multi-Dconv Head Transposed Attention (MDTA) and Gated Feed-Forward Network (GDFN) Restormer Transformer on Kaggle Dual GPUs using the 7-combination degraded dataset.

### Recommended Accelerator: Tesla T4 Dual GPU (GPU T4 x 2)
- Multi-GPU DataParallel Support: Automatically detects and utilizes all available GPUs (GPU 0 + GPU 1).

### Key Implementation Details:
- Architecture: Transformer for Image Restoration (Restormer) with MDTA channel-transposed attention and GDFN feed-forward blocks.
- Robust Dataset Auto-Discovery: Dynamically discovers root dataset path and subfolder structures on Kaggle.
- Multi-GPU Execution: Wrapped with torch.nn.DataParallel across multiple GPUs.
- Cell Execution Timers: Tracks individual cell execution time and cumulative elapsed time.
- Progress Monitoring: Standard tqdm progress bars with live epoch loss, PSNR, and SSIM.
- Training Duration: Pre-configured for 5 initial epochs.
- Model Checkpointing: Saves transformer_model.pth to the output directory.
- Results Export: Packages transformer_model.pth, training_metrics.json, training curves, and test sample visualizations into transformer_restoration_results.zip.

In [ ]:
import time
GLOBAL_START_TIME = time.time()
_cell_start = time.time()

import os
import glob
import json
import math
import random
import zipfile
from pathlib import Path
from PIL import Image
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

# Set seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def print_timer(cell_name, cell_start):
    cell_elapsed = time.time() - cell_start
    total_elapsed = time.time() - GLOBAL_START_TIME
    print(f"[Cell Timer - {cell_name}] Cell Time: {cell_elapsed:.2f}s | Cumulative Time: {total_elapsed:.2f}s")

print_timer("Imports and Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Environment and Dataset Path Auto-Discovery Resolution
def find_degraded_dataset_root():
    search_candidates = [
        "/kaggle/input/datasets/hetulkadiya/degraded-dataset-zip-file/degraded_dataset",
        "/kaggle/input/degraded-dataset-zip-file/degraded_dataset",
        "/kaggle/input/degraded_dataset",
        "../data/degraded_dataset",
        "./degraded_dataset"
    ]
    for cand in search_candidates:
        if os.path.exists(cand):
            return cand
            
    # Dynamic search for any directory containing train/val or clean folders
    for search_base in ["/kaggle/input", "..", "."]:
        if os.path.exists(search_base):
            for root, dirs, files in os.walk(search_base):
                if "clean" in dirs or "c1_only_noise" in dirs or ("train" in dirs and "val" in dirs):
                    return root
                    
    return search_candidates[0]

DATASET_ROOT = find_degraded_dataset_root()
OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "."

# Multi-GPU Accelerator Setup
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

def get_compute_device():
    if NUM_GPUS > 0:
        try:
            for i in range(NUM_GPUS):
                gpu_name = torch.cuda.get_device_name(i)
                print(f"Detected GPU {i}: {gpu_name}")
            
            test_tensor = torch.randn(2, 3, 16, 16, device="cuda:0")
            test_conv = nn.Conv2d(3, 16, 3, padding=1).to("cuda:0")
            _ = test_conv(test_tensor)
            return torch.device("cuda")
        except Exception as e:
            print(f"Warning: CUDA error encountered on current GPU ({e}).")
            print("Recommendation: In Kaggle Notebook Settings (right sidebar), switch Accelerator to 'GPU T4 x 2'.")
            return torch.device("cpu")
    return torch.device("cpu")

DEVICE = get_compute_device()
print(f"Active Main Device: {DEVICE} (Total GPUs Available: {NUM_GPUS})")
print(f"Dataset Root: {DATASET_ROOT}")
print(f"Output Directory: {OUTPUT_DIR}")

print_timer("Path and Multi-GPU Device Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Robust PyTorch Dataset for Paired Clean and 7-Combo Degraded Images
class KaggleDegradedDataset(Dataset):
    def __init__(self, root_dir, split="train", target_size=(128, 128)):
        self.target_size = target_size
        self.pairs = []
        
        # Discover split directory
        split_dir = os.path.join(root_dir, split)
        if not os.path.exists(split_dir):
            # Fallback search for split folder case-insensitively
            candidates = [d for d in os.listdir(root_dir) if d.lower() == split.lower() and os.path.isdir(os.path.join(root_dir, d))]
            if candidates:
                split_dir = os.path.join(root_dir, candidates[0])
            else:
                split_dir = root_dir
                
        # Discover clean ground truth directory
        clean_dir = None
        for clean_name in ["clean", "ground_truth", "gt", "clean_images", "original"]:
            test_path = os.path.join(split_dir, clean_name)
            if os.path.exists(test_path):
                clean_dir = test_path
                break
                
        if clean_dir is None:
            # Dynamic search for any directory named clean inside split_dir
            for r, dirs, files in os.walk(split_dir):
                for d in dirs:
                    if d.lower() in ["clean", "ground_truth", "gt"]:
                        clean_dir = os.path.join(r, d)
                        break
                if clean_dir:
                    break
                    
        combos = [
            "c1_only_noise", "c2_only_blur", "c3_only_low_light",
            "c4_noise_blur", "c5_noise_low_light", "c6_blur_low_light",
            "c7_all_three"
        ]
        
        if clean_dir and os.path.exists(clean_dir):
            clean_files = sorted(glob.glob(os.path.join(clean_dir, "*.[pP][nN][gG]")) + \
                                 glob.glob(os.path.join(clean_dir, "*.[jJ][pP][gG]")) + \
                                 glob.glob(os.path.join(clean_dir, "*.[jJ][pP][eE][gG]")))
                                 
            parent_dir = os.path.dirname(clean_dir)
            subfolders = [d for d in os.listdir(parent_dir) if os.path.isdir(os.path.join(parent_dir, d)) and d != os.path.basename(clean_dir)]
            
            for clean_path in clean_files:
                filename = os.path.basename(clean_path)
                # Try matched combos first
                for combo in combos:
                    deg_path = os.path.join(parent_dir, combo, filename)
                    if os.path.exists(deg_path):
                        self.pairs.append((deg_path, clean_path, combo))
                # If specific combo names don't match, check all sibling degraded subfolders
                if len(self.pairs) == 0:
                    for sub in subfolders:
                        deg_path = os.path.join(parent_dir, sub, filename)
                        if os.path.exists(deg_path):
                            self.pairs.append((deg_path, clean_path, sub))
                            
        self.to_tensor = T.Compose([
            T.Resize(target_size),
            T.ToTensor()
        ])
        print(f"Loaded [{split.upper()}] dataset split from '{split_dir}': {len(self.pairs)} paired samples.")

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        deg_path, clean_path, combo = self.pairs[idx]
        with Image.open(deg_path) as deg_img, Image.open(clean_path) as clean_img:
            deg_tensor = self.to_tensor(deg_img.convert("RGB"))
            clean_tensor = self.to_tensor(clean_img.convert("RGB"))
        return deg_tensor, clean_tensor

# Create Datasets and DataLoaders
USE_CUDA = (DEVICE.type == "cuda")
BATCH_SIZE = 32 if NUM_GPUS > 1 else (16 if USE_CUDA else 4)
NUM_WORKERS = (2 * NUM_GPUS) if USE_CUDA else 0

train_dataset = KaggleDegradedDataset(DATASET_ROOT, split="train")
val_dataset = KaggleDegradedDataset(DATASET_ROOT, split="val")
test_dataset = KaggleDegradedDataset(DATASET_ROOT, split="test")

# Fallback check if dataset root is nested
if len(train_dataset) == 0:
    print("Warning: Initial dataset discovery returned 0 samples. Executing deep dynamic path search...")
    for root, dirs, files in os.walk(DATASET_ROOT):
        if "clean" in dirs:
            DATASET_ROOT = os.path.dirname(root) if os.path.basename(root) in ["train", "val", "test"] else root
            print(f"Updated DATASET_ROOT to: {DATASET_ROOT}")
            train_dataset = KaggleDegradedDataset(DATASET_ROOT, split="train")
            val_dataset = KaggleDegradedDataset(DATASET_ROOT, split="val")
            test_dataset = KaggleDegradedDataset(DATASET_ROOT, split="test")
            break

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=USE_CUDA)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=USE_CUDA)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=USE_CUDA)

print(f"DataLoader Configured: Batch Size = {BATCH_SIZE} | Workers = {NUM_WORKERS}")
print(f"Dataset Pair Counts -> Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")
print_timer("Dataset and DataLoaders Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Model 3: Transformer Architecture for Image Restoration (Restormer)
class MDTA(nn.Module):
    """Multi-Dconv Head Transposed Self-Attention (MDTA) module."""
    def __init__(self, dim, num_heads=4, bias=False):
        super(MDTA, self).__init__()
        self.num_heads = num_heads
        self.temperature = nn.Parameter(torch.ones(num_heads, 1, 1))

        self.qkv = nn.Conv2d(dim, dim * 3, kernel_size=1, bias=bias)
        self.qkv_dwconv = nn.Conv2d(dim * 3, dim * 3, kernel_size=3, stride=1, padding=1, groups=dim * 3, bias=bias)
        self.project_out = nn.Conv2d(dim, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        b, c, h, w = x.shape
        qkv = self.qkv_dwconv(self.qkv(x))
        q, k, v = qkv.chunk(3, dim=1)
        
        q = q.reshape(b, self.num_heads, -1, h * w)
        k = k.reshape(b, self.num_heads, -1, h * w)
        v = v.reshape(b, self.num_heads, -1, h * w)
        
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)
        
        attn = (q @ k.transpose(-2, -1)) * self.temperature
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v)
        out = out.reshape(b, c, h, w)
        out = self.project_out(out)
        return out

class GDFN(nn.Module):
    """Gated-Dconv Feed-Forward Network (GDFN) module."""
    def __init__(self, dim, ffn_expansion_factor=2.66, bias=False):
        super(GDFN, self).__init__()
        hidden_features = int(dim * ffn_expansion_factor)
        self.project_in = nn.Conv2d(dim, hidden_features * 2, kernel_size=1, bias=bias)
        self.dwconv = nn.Conv2d(hidden_features * 2, hidden_features * 2, kernel_size=3, stride=1, padding=1, groups=hidden_features * 2, bias=bias)
        self.project_out = nn.Conv2d(hidden_features, dim, kernel_size=1, bias=bias)

    def forward(self, x):
        x = self.project_in(x)
        x1, x2 = self.dwconv(x).chunk(2, dim=1)
        x = F.gelu(x1) * x2
        x = self.project_out(x)
        return x

class LayerNorm2d(nn.Module):
    def __init__(self, channels, eps=1e-6):
        super(LayerNorm2d, self).__init__()
        self.weight = nn.Parameter(torch.ones(1, channels, 1, 1))
        self.bias = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.eps = eps

    def forward(self, x):
        u = x.mean(1, keepdim=True)
        s = (x - u).pow(2).mean(1, keepdim=True)
        x = (x - u) / torch.sqrt(s + self.eps)
        x = self.weight * x + self.bias
        return x

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads=4, ffn_expansion_factor=2.66, bias=False):
        super(TransformerBlock, self).__init__()
        self.norm1 = LayerNorm2d(dim)
        self.attn = MDTA(dim, num_heads, bias)
        self.norm2 = LayerNorm2d(dim)
        self.ffn = GDFN(dim, ffn_expansion_factor, bias)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

class Restormer(nn.Module):
    """
    Model 3: Transformer-based Image Restoration Network (Restormer).
    Operates with Multi-Dconv Head Transposed Self-Attention and Gated Feed-Forward blocks.
    """
    def __init__(self, in_channels=3, out_channels=3, dim=48, num_blocks=[2, 2, 2, 2], heads=[1, 2, 4, 8]):
        super(Restormer, self).__init__()
        self.patch_embed = nn.Conv2d(in_channels, dim, kernel_size=3, padding=1)

        # Encoder Level 1
        self.encoder_level1 = nn.Sequential(*[TransformerBlock(dim=dim, num_heads=heads[0]) for _ in range(num_blocks[0])])
        self.down1_2 = nn.Conv2d(dim, dim * 2, kernel_size=4, stride=2, padding=1)
        
        # Encoder Level 2
        self.encoder_level2 = nn.Sequential(*[TransformerBlock(dim=dim * 2, num_heads=heads[1]) for _ in range(num_blocks[1])])

        # Bottleneck
        self.bottleneck = nn.Sequential(*[TransformerBlock(dim=dim * 2, num_heads=heads[2]) for _ in range(num_blocks[2])])

        # Decoder Level 1
        self.up2_1 = nn.ConvTranspose2d(dim * 2, dim, kernel_size=4, stride=2, padding=1)
        self.reduce_chan = nn.Conv2d(dim * 2, dim, kernel_size=1)
        self.decoder_level1 = nn.Sequential(*[TransformerBlock(dim=dim, num_heads=heads[0]) for _ in range(num_blocks[3])])

        self.output = nn.Sequential(
            nn.Conv2d(dim, out_channels, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, inp_img):
        feat = self.patch_embed(inp_img)
        e1 = self.encoder_level1(feat)
        d1 = self.down1_2(e1)
        
        e2 = self.encoder_level2(d1)
        b = self.bottleneck(e2)
        
        u1 = self.up2_1(b)
        u1 = self.reduce_chan(torch.cat([u1, e1], dim=1))
        d_out = self.decoder_level1(u1)
        
        out = self.output(d_out + feat)
        return out

base_model = Restormer()

# Wrap model with DataParallel if multiple GPUs are detected
if NUM_GPUS > 1:
    print(f"Enabling Multi-GPU DataParallel across {NUM_GPUS} GPUs!")
    model = nn.DataParallel(base_model)
else:
    model = base_model

model = model.to(DEVICE)

num_params = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
print(f"Restormer Transformer Model initialized. Total Trainable Parameters: {num_params:,}")

print_timer("Multi-GPU Transformer Model Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Loss Function and Metric Evaluation Functions (PSNR and SSIM)
class CombinedLoss(nn.Module):
    def __init__(self, l1_weight=0.8, mse_weight=0.2):
        super(CombinedLoss, self).__init__()
        self.l1_weight = l1_weight
        self.mse_weight = mse_weight
        self.l1 = nn.L1Loss()
        self.mse = nn.MSELoss()

    def forward(self, pred, target):
        return self.l1_weight * self.l1(pred, target) + self.mse_weight * self.mse(pred, target)

def calculate_batch_psnr(pred, target):
    with torch.no_grad():
        mse = torch.mean((pred - target) ** 2, dim=[1, 2, 3])
        psnr = 10.0 * torch.log10(1.0 / (mse + 1e-8))
        return torch.mean(psnr).item()

def calculate_batch_ssim(pred, target):
    with torch.no_grad():
        mu_x = F.avg_pool2d(pred, 3, 1, 1)
        mu_y = F.avg_pool2d(target, 3, 1, 1)
        sigma_x = F.avg_pool2d(pred * pred, 3, 1, 1) - mu_x ** 2
        sigma_y = F.avg_pool2d(target * target, 3, 1, 1) - mu_y ** 2
        sigma_xy = F.avg_pool2d(pred * target, 3, 1, 1) - mu_x * mu_y
        C1, C2 = 0.01 ** 2, 0.03 ** 2
        ssim_map = ((2 * mu_x * mu_y + C1) * (2 * sigma_xy + C2)) / ((mu_x ** 2 + mu_y ** 2 + C1) * (sigma_x + sigma_y + C2))
        return torch.mean(ssim_map).item()

def extract_clean_state_dict(m):
    return m.module.state_dict() if isinstance(m, nn.DataParallel) else m.state_dict()

def load_clean_state_dict(m, path, device):
    state_dict = torch.load(path, map_location=device)
    if isinstance(m, nn.DataParallel):
        m.module.load_state_dict(state_dict)
    else:
        m.load_state_dict(state_dict)

criterion = CombinedLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, betas=(0.9, 0.999))

print_timer("Loss and Metric Functions Setup", _cell_start)

In [ ]:
_cell_start = time.time()

# Multi-GPU Epoch Training and Validation Loop
EPOCHS = 5  # Initial training duration set to 5 epochs

history = {
    "train_loss": [],
    "val_loss": [],
    "val_psnr": [],
    "val_ssim": []
}

best_val_psnr = 0.0
MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "transformer_model.pth")

print(f"Starting Restormer Transformer Training for {EPOCHS} Epochs across {max(NUM_GPUS, 1)} device(s)...")

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    
    # Training Phase
    model.train()
    train_loss = 0.0
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=True)
    
    for degraded, clean in train_pbar:
        degraded, clean = degraded.to(DEVICE), clean.to(DEVICE)
        optimizer.zero_grad()
        output = model(degraded)
        loss = criterion(output, clean)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
        train_pbar.set_postfix({"Loss": f"{loss.item():.4f}"})
        
    train_loss /= max(len(train_loader), 1)
    history["train_loss"].append(round(train_loss, 4))
    
    # Validation Phase
    model.eval()
    val_loss = 0.0
    val_psnr = 0.0
    val_ssim = 0.0
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch}/{EPOCHS} [Val]  ", leave=False)
    
    with torch.no_grad():
        for degraded, clean in val_pbar:
            degraded, clean = degraded.to(DEVICE), clean.to(DEVICE)
            restored = model(degraded)
            loss = criterion(restored, clean)
            val_loss += loss.item()
            val_psnr += calculate_batch_psnr(restored, clean)
            val_ssim += calculate_batch_ssim(restored, clean)
            
    val_loss /= max(len(val_loader), 1)
    val_psnr /= max(len(val_loader), 1)
    val_ssim /= max(len(val_loader), 1)
    
    history["val_loss"].append(round(val_loss, 4))
    history["val_psnr"].append(round(val_psnr, 2))
    history["val_ssim"].append(round(val_ssim, 4))
    
    epoch_time = time.time() - epoch_start
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} [{epoch_time:.1f}s] -> Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | Val SSIM: {val_ssim:.4f}")
    
    if val_psnr > best_val_psnr:
        best_val_psnr = val_psnr
        clean_sd = extract_clean_state_dict(model)
        torch.save(clean_sd, MODEL_SAVE_PATH)
        print(f"  --> Saved best checkpoint to '{MODEL_SAVE_PATH}' (Val PSNR: {val_psnr:.2f} dB)")

print_timer("5-Epoch Multi-GPU Training Loop", _cell_start)

In [ ]:
_cell_start = time.time()

# Final Evaluation and Benchmark on Test Set
if os.path.exists(MODEL_SAVE_PATH):
    load_clean_state_dict(model, MODEL_SAVE_PATH, DEVICE)

model.eval()
deg_psnr_total, deg_ssim_total = 0.0, 0.0
res_psnr_total, res_ssim_total = 0.0, 0.0

with torch.no_grad():
    for degraded, clean in tqdm(test_loader, desc="Evaluating Test Set"):
        degraded, clean = degraded.to(DEVICE), clean.to(DEVICE)
        restored = model(degraded)
        
        deg_psnr_total += calculate_batch_psnr(degraded, clean)
        deg_ssim_total += calculate_batch_ssim(degraded, clean)
        
        res_psnr_total += calculate_batch_psnr(restored, clean)
        res_ssim_total += calculate_batch_ssim(restored, clean)

n_test_batches = max(len(test_loader), 1)
avg_deg_psnr = deg_psnr_total / n_test_batches
avg_deg_ssim = deg_ssim_total / n_test_batches
avg_res_psnr = res_psnr_total / n_test_batches
avg_res_ssim = res_ssim_total / n_test_batches

psnr_gain = avg_res_psnr - avg_deg_psnr
ssim_gain = avg_res_ssim - avg_deg_ssim

print("\n======================================================")
print("MODEL 3 (RESTORMER TRANSFORMER) TEST BENCHMARK RESULTS")
print("======================================================")
print(f"GPUs Utilized           : {max(NUM_GPUS, 1)}")
print(f"Degraded Input Quality  : PSNR = {avg_deg_psnr:.2f} dB | SSIM = {avg_deg_ssim:.4f}")
print(f"Transformer Restored    : PSNR = {avg_res_psnr:.2f} dB | SSIM = {avg_res_ssim:.4f}")
print(f"Restoration dB Gain     : PSNR Gain = +{psnr_gain:.2f} dB | SSIM Gain = +{ssim_gain:.4f}")
print("======================================================")

# Save Evaluation Metrics JSON
metrics_json_path = os.path.join(OUTPUT_DIR, "training_metrics.json")
metrics_data = {
    "model_name": "Restormer Transformer Restoration Network",
    "gpus_used": max(NUM_GPUS, 1),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "loss_functions": ["L1Loss", "MSELoss"],
    "optimizer": "Adam (lr=1e-4)",
    "history": history,
    "test_results": {
        "degraded_psnr": round(avg_deg_psnr, 2),
        "degraded_ssim": round(avg_deg_ssim, 4),
        "restored_psnr": round(avg_res_psnr, 2),
        "restored_ssim": round(avg_res_ssim, 4),
        "psnr_gain_db": round(psnr_gain, 2),
        "ssim_gain": round(ssim_gain, 4)
    }
}
with open(metrics_json_path, "w") as f:
    json.dump(metrics_data, f, indent=2)

print_timer("Test Evaluation and Metrics Export", _cell_start)

In [ ]:
_cell_start = time.time()

# Plot Training Curves and Generate Sample Visual Restorations
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss Curve
axes[0].plot(range(1, EPOCHS + 1), history["train_loss"], label="Train Loss", marker="o", color="#ef4444")
axes[0].plot(range(1, EPOCHS + 1), history["val_loss"], label="Val Loss", marker="s", color="#3b82f6")
axes[0].set_title("L1 + MSE Loss Curve", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].grid(True, linestyle="--", alpha=0.5)
axes[0].legend()

# PSNR Curve
axes[1].plot(range(1, EPOCHS + 1), history["val_psnr"], label="Val PSNR (dB)", marker="^", color="#10b981")
axes[1].set_title("Validation PSNR (dB)", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PSNR (dB)")
axes[1].grid(True, linestyle="--", alpha=0.5)
axes[1].legend()

# SSIM Curve
axes[2].plot(range(1, EPOCHS + 1), history["val_ssim"], label="Val SSIM", marker="d", color="#8b5cf6")
axes[2].set_title("Validation SSIM Score", fontsize=12, fontweight="bold")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("SSIM Score")
axes[2].grid(True, linestyle="--", alpha=0.5)
axes[2].legend()

plt.tight_layout()
plots_image_path = os.path.join(OUTPUT_DIR, "loss_psnr_ssim_curves.png")
plt.savefig(plots_image_path, dpi=300)
plt.show()

# Visual Restoration Samples Plot
model.eval()
for deg_b, clean_b in test_loader:
    deg_b, clean_b = deg_b.to(DEVICE), clean_b.to(DEVICE)
    with torch.no_grad():
        res_b = model(deg_b)
    break

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i in range(min(3, deg_b.shape[0])):
    deg_np = deg_b[i].cpu().permute(1, 2, 0).numpy()
    res_np = res_b[i].cpu().permute(1, 2, 0).numpy()
    clean_np = clean_b[i].cpu().permute(1, 2, 0).numpy()
    
    axes[i, 0].imshow(clean_np)
    axes[i, 0].set_title("Original Clean", fontsize=9)
    axes[i, 0].axis("off")
    
    axes[i, 1].imshow(deg_np)
    axes[i, 1].set_title("Degraded Input", fontsize=9)
    axes[i, 1].axis("off")
    
    axes[i, 2].imshow(res_np)
    axes[i, 2].set_title("Transformer Restored", fontsize=9)
    axes[i, 2].axis("off")

plt.suptitle("Restormer Transformer Test Sample Restorations", fontsize=13)
plt.tight_layout()
samples_image_path = os.path.join(OUTPUT_DIR, "test_restoration_samples.png")
plt.savefig(samples_image_path, dpi=300)
plt.show()

print_timer("Plots and Visual Restorations", _cell_start)

In [ ]:
_cell_start = time.time()

# Package Checkpoints, Metrics, Plots into transformer_restoration_results.zip
ZIP_FILE_PATH = os.path.join(OUTPUT_DIR, "transformer_restoration_results.zip")

files_to_zip = [
    (MODEL_SAVE_PATH, "transformer_model.pth"),
    (metrics_json_path, "training_metrics.json"),
    (plots_image_path, "loss_psnr_ssim_curves.png"),
    (samples_image_path, "test_restoration_samples.png")
]

with zipfile.ZipFile(ZIP_FILE_PATH, "w", zipfile.ZIP_DEFLATED) as zipf:
    for src_path, arcname in files_to_zip:
        if os.path.exists(src_path):
            zipf.write(src_path, arcname=arcname)
            print(f"Added to Zip: {arcname}")

print(f"\n======================================================")
print(f"RESULTS ZIP ARCHIVE CREATED SUCCESSFULLY")
print(f"Zip Path: {ZIP_FILE_PATH}")
print(f"Size: {os.path.getsize(ZIP_FILE_PATH) / (1024 * 1024):.2f} MB")
print(f"Contains: transformer_model.pth, training_metrics.json, curves, visual samples")
print(f"======================================================")

print_timer("Zip Packaging and Export", _cell_start)